EXTRACTING DATA FROM KAGGLE

In [110]:
from pathlib import Path 
import pandas as pd
import os

# print("Current working directory:", os.getcwd())

In [111]:
# store the different dataframes into 1 dictionary
dataframes = {}
encodings_to_try = ["utf-8", "latin-1", "cp1252"]

In [112]:
# loop over files in the folder and get all the csv files
folder_path = Path('../data/raw/f1')
csv_files = list(folder_path.glob("*.csv"))

# for each CSV found, create a df using the frame and store it in the dataframes dictionary
for file_path in csv_files:
    # print(file_path)
    for enc in encodings_to_try:
        try: 
            # try to read csv using the encodings listed
            df = pd.read_csv(f"{file_path}", encoding=enc)
            
            # if successful, store to dataframe with the stem as the key
            dataframes[file_path.stem] = df
            print(f"Loaded {file_path.stem} into dataframe successfully with encoding {enc}")
            break
        
        except UnicodeDecodeError: 
            continue

        except Exception as e:
            # catch other types of errors
            print(f"Failed to load {file_path.stem} {e}")
            break

Loaded circuits into dataframe successfully with encoding latin-1
Loaded constructors into dataframe successfully with encoding utf-8
Loaded constructor_standings into dataframe successfully with encoding utf-8
Loaded drivers into dataframe successfully with encoding utf-8
Loaded driver_standings into dataframe successfully with encoding utf-8
Loaded f1_2025_last_race_results into dataframe successfully with encoding utf-8
Loaded qualifying into dataframe successfully with encoding utf-8
Loaded races into dataframe successfully with encoding utf-8
Loaded results into dataframe successfully with encoding utf-8


In [113]:
# display all loaded CSVs using the keys
print(f"All loaded CSVs: {dataframes.keys()}")

All loaded CSVs: dict_keys(['circuits', 'constructors', 'constructor_standings', 'drivers', 'driver_standings', 'f1_2025_last_race_results', 'qualifying', 'races', 'results'])


INSPECTING THE DATA

In [114]:
# get 1 dataframe and then try to inspect the data
dataframes['circuits'].head()

,circuit_id,name,lat,long,locality,country,Wikipedia_url
0,silverstone,Silverstone Circuit,52.0786,-1.01694,Silverstone,UK,https://en.wikipedia.org/wiki/Silverstone_Circuit
1,monaco,Circuit de Monaco,43.7347,7.42056,Monte-Carlo,Monaco,https://en.wikipedia.org/wiki/Circuit_de_Monaco
2,indianapolis,Indianapolis Motor Speedway,39.7950,-86.23470,Indianapolis,USA,https://en.wikipedia.org/wiki/Indianapolis_Mot...
3,bremgarten,Circuit Bremgarten,46.9589,7.40194,Bern,Switzerland,https://en.wikipedia.org/wiki/Circuit_Bremgarten
4,spa,Circuit de Spa-Francorchamps,50.4372,5.97139,Spa,Belgium,https://en.wikipedia.org/wiki/Circuit_de_Spa-F...


In [115]:
# check the total number of null values in the table
dataframes['circuits'].isnull().sum()

circuit_id        0
name              0
lat               0
long              0
locality          0
country           0
Wikipedia_url     0
dtype: int64

In [116]:
# # display all df heads
# for key, df in dataframes.items():
#     print(f"===={key}====")
#     display(df.head())
    
#     print(f"Total number of Null values in {key}")
#     if 'season' in df.columns: 
#         filtered_df = df[df['season'] >= 2015]
#         print("Min season after filter: ", filtered_df['season'].min())
#         print("Max season after filter: ", filtered_df['season'].max())

#         display(filtered_df.isnull().sum())
#     else: 
#         display(df.isnull().sum())

CLEANING THE DATA - Circuits

In [117]:
# Clean the circuit data

circuits_table = dataframes["circuits"]
circuits_table.head()


,circuit_id,name,lat,long,locality,country,Wikipedia_url
0,silverstone,Silverstone Circuit,52.0786,-1.01694,Silverstone,UK,https://en.wikipedia.org/wiki/Silverstone_Circuit
1,monaco,Circuit de Monaco,43.7347,7.42056,Monte-Carlo,Monaco,https://en.wikipedia.org/wiki/Circuit_de_Monaco
2,indianapolis,Indianapolis Motor Speedway,39.7950,-86.23470,Indianapolis,USA,https://en.wikipedia.org/wiki/Indianapolis_Mot...
3,bremgarten,Circuit Bremgarten,46.9589,7.40194,Bern,Switzerland,https://en.wikipedia.org/wiki/Circuit_Bremgarten
4,spa,Circuit de Spa-Francorchamps,50.4372,5.97139,Spa,Belgium,https://en.wikipedia.org/wiki/Circuit_de_Spa-F...


In [118]:
# Check datatypes of the columns to verify 
datatypes = circuits_table.dtypes

# print datatypes of each column 
datatypes

circuit_id            str
name                  str
lat               float64
long              float64
locality              str
country               str
Wikipedia_url         str
dtype: object

In [119]:
# check column names 
circuits_table.columns.tolist()

['circuit_id', 'name', 'lat', 'long', 'locality', 'country', 'Wikipedia_url ']

In [120]:
# remove whitespace from names 
circuits_table.columns = circuits_table.columns.str.strip()

# drop the wikipedia column 
cleaned_circuits_table = circuits_table.drop(columns=['Wikipedia_url'])

# display new table 
cleaned_circuits_table

,circuit_id,name,lat,long,locality,country
0,silverstone,Silverstone Circuit,52.0786,-1.01694,Silverstone,UK
1,monaco,Circuit de Monaco,43.7347,7.42056,Monte-Carlo,Monaco
2,indianapolis,Indianapolis Motor Speedway,39.7950,-86.23470,Indianapolis,USA
3,bremgarten,Circuit Bremgarten,46.9589,7.40194,Bern,Switzerland
4,spa,Circuit de Spa-Francorchamps,50.4372,5.97139,Spa,Belgium
...,...,...,...,...,...,...
71,mugello,Autodromo Internazionale del Mugello,43.9975,11.37190,Mugello,Italy
72,portimao,Autódromo Internacional do Algarve,37.2270,-8.62670,PortimÃ£o,Portugal
73,losail,Losail International Circuit,25.4900,51.45420,Al Daayen,Qatar
74,jeddah,Jeddah Corniche Circuit,21.6319,39.10440,Jeddah,Saudi Arabia


In [121]:
# check for duplicates 
count_of_duplicates = cleaned_circuits_table.duplicated().sum()
print(f"duplicate count: {count_of_duplicates}")

# check for null values 
count_null_values = cleaned_circuits_table.isna().sum()
print(count_null_values)

duplicate count: 0
circuit_id    0
name          0
lat           0
long          0
locality      0
country       0
dtype: int64


CLEANING THE DATA: CONSTRUCTORS

In [122]:
# Clean the constructors data

constructors_table = dataframes["constructors"]
constructors_table.head()


,constructor_id,name,nationality,Wikipedia_url
0,alfa,Alfa Romeo,Swiss,https://en.wikipedia.org/wiki/Alfa_Romeo_in_Fo...
1,lago,Talbot-Lago,French,https://en.wikipedia.org/wiki/Talbot-Lago_T26C
2,era,ERA,British,http://en.espn.co.uk/era/motorsport/team/292.h...
3,maserati,Maserati,Italian,https://en.wikipedia.org/wiki/Maserati_in_moto...
4,alta,Alta,British,https://f1.fandom.com/wiki/Alta_Car_and_Engine...


In [123]:
# Check datatypes of the columns to verify 
datatypes = constructors_table.dtypes

# print datatypes of each column 
datatypes

constructor_id    str
name              str
nationality       str
Wikipedia_url     str
dtype: object

In [124]:
# remove whitespace from names 
constructors_table.columns = constructors_table.columns.str.strip()

# drop the wikipedia column 
cleaned_constructors_table = circuits_table.drop(columns=['Wikipedia_url'])

# display new table 
cleaned_constructors_table

,circuit_id,name,lat,long,locality,country
0,silverstone,Silverstone Circuit,52.0786,-1.01694,Silverstone,UK
1,monaco,Circuit de Monaco,43.7347,7.42056,Monte-Carlo,Monaco
2,indianapolis,Indianapolis Motor Speedway,39.7950,-86.23470,Indianapolis,USA
3,bremgarten,Circuit Bremgarten,46.9589,7.40194,Bern,Switzerland
4,spa,Circuit de Spa-Francorchamps,50.4372,5.97139,Spa,Belgium
...,...,...,...,...,...,...
71,mugello,Autodromo Internazionale del Mugello,43.9975,11.37190,Mugello,Italy
72,portimao,Autódromo Internacional do Algarve,37.2270,-8.62670,PortimÃ£o,Portugal
73,losail,Losail International Circuit,25.4900,51.45420,Al Daayen,Qatar
74,jeddah,Jeddah Corniche Circuit,21.6319,39.10440,Jeddah,Saudi Arabia


In [125]:
# check for duplicates 
count_of_duplicates = cleaned_constructors_table.duplicated().sum()
print(f"duplicate count: {count_of_duplicates}")

# check for null values 
count_null_values = cleaned_constructors_table.isna().sum()
print(count_null_values)

duplicate count: 0
circuit_id    0
name          0
lat           0
long          0
locality      0
country       0
dtype: int64


CLEANING THE DATA: constructor standings

In [126]:
# Clean the constructor standings data

constructors_standings_table = dataframes["constructor_standings"]
constructors_standings_table.head()

,season,round,constructor_id,position,points,wins
0,1950,7,alfa,NaN,0.0,6
1,1950,7,kurtis_kraft,NaN,0.0,1
2,1950,7,ferrari,NaN,0.0,0
3,1950,7,deidt,NaN,0.0,0
4,1950,7,lago,NaN,0.0,0


In [127]:
# Check datatypes of the columns to verify 
datatypes = constructors_standings_table.dtypes

# print datatypes of each column 
datatypes

season              int64
round               int64
constructor_id        str
position          float64
points            float64
wins                int64
dtype: object

In [128]:
# filter the season to only 2015 and above 
filtered_standings = constructors_standings_table[constructors_standings_table['season'] >= 2015]

# display filtered (2015-2025)
filtered_standings

,season,round,constructor_id,position,points,wins
950,2015,19,mercedes,1.0,703.0,16
951,2015,19,ferrari,2.0,428.0,3
952,2015,19,williams,3.0,257.0,0
953,2015,19,red_bull,4.0,187.0,0
954,2015,19,force_india,5.0,136.0,0
...,...,...,...,...,...,...
1056,2025,21,rb,6.0,82.0,0
1057,2025,21,aston_martin,7.0,72.0,0
1058,2025,21,haas,8.0,70.0,0
1059,2025,21,sauber,9.0,62.0,0


In [129]:
# check for duplicates 
count_of_duplicates = filtered_standings.duplicated().sum()
print(f"duplicate count: {count_of_duplicates}")

# check for null values 
count_null_values = filtered_standings.isna().sum()
print(count_null_values)

duplicate count: 0
season            0
round             0
constructor_id    0
position          0
points            0
wins              0
dtype: int64


CLEANING THE DATA: DRIVERS

In [130]:
# Clean the drivers data

drivers_table = dataframes["drivers"]
drivers_table.head()

,driver_id,givenName,familyName,nationality,dob
0,farina,Nino,Farina,Italian,1906-10-30
1,fagioli,Luigi,Fagioli,Italian,1898-06-09
2,reg_parnell,Reg,Parnell,British,1911-07-02
3,cabantous,Yves,Cabantous,French,1904-10-08
4,rosier,Louis,Rosier,French,1905-11-05


In [131]:
# Check datatypes of the columns to verify 
datatypes = drivers_table.dtypes

# print datatypes of each column 
datatypes

driver_id      str
givenName      str
familyName     str
nationality    str
dob            str
dtype: object

In [132]:
# check for duplicates 
count_of_duplicates = drivers_table.duplicated().sum()
print(f"duplicate count: {count_of_duplicates}")

# check for null values 
count_null_values = drivers_table.isna().sum()
print(count_null_values)

duplicate count: 0
driver_id      0
givenName      0
familyName     0
nationality    0
dob            0
dtype: int64


In [133]:
# rename column of givenName and familyName
drivers_table = drivers_table.rename(columns={'givenName':'given_name', 'familyName':'family_name'})
drivers_table

,driver_id,given_name,family_name,nationality,dob
0,farina,Nino,Farina,Italian,1906-10-30
1,fagioli,Luigi,Fagioli,Italian,1898-06-09
2,reg_parnell,Reg,Parnell,British,1911-07-02
3,cabantous,Yves,Cabantous,French,1904-10-08
4,rosier,Louis,Rosier,French,1905-11-05
...,...,...,...,...,...
611,antonelli,Andrea Kimi,Antonelli,Italian,2006-08-25
612,lawson,Liam,Lawson,New Zealander,2002-02-11
613,bortoleto,Gabriel,Bortoleto,Brazilian,2004-10-14
614,doohan,Jack,Doohan,Australian,2003-01-20


CLEANING THE DATA: DRIVER STANDINGS

In [134]:
# Clean the driver standings data

drivers_standings_table = dataframes["driver_standings"]
drivers_standings_table.head()

,season,round,driver_id,position,points,wins
0,1950,7,farina,1.0,30.0,3
1,1950,7,fangio,2.0,27.0,3
2,1950,7,fagioli,3.0,24.0,0
3,1950,7,rosier,4.0,13.0,0
4,1950,7,ascari,5.0,11.0,0


In [135]:
# Check datatypes of the columns to verify 
datatypes = drivers_standings_table.dtypes

# print datatypes of each column 
datatypes

season         int64
round          int64
driver_id        str
position     float64
points       float64
wins           int64
dtype: object

In [136]:
# filter the driver standings season to 2015 - 2025
filtered_standings = drivers_standings_table[drivers_standings_table['season'] >= 2015]
filtered_standings

,season,round,driver_id,position,points,wins
2887,2015,19,hamilton,1.0,381.0,10
2888,2015,19,rosberg,2.0,322.0,6
2889,2015,19,vettel,3.0,278.0,3
2890,2015,19,raikkonen,4.0,150.0,0
2891,2015,19,bottas,5.0,136.0,0
...,...,...,...,...,...,...
3126,2025,21,tsunoda,17.0,32.0,0
3127,2025,21,gasly,18.0,22.0,0
3128,2025,21,bortoleto,19.0,19.0,0
3129,2025,21,colapinto,20.0,0.0,0


In [137]:
# check for duplicates 
count_of_duplicates = filtered_standings.duplicated().sum()
print(f"duplicate count: {count_of_duplicates}")

# check for null values 
count_null_values = filtered_standings.isna().sum()
print(count_null_values)

# null position must be because driver wasnt officially lsited and did nto participate
null_positions = filtered_standings[filtered_standings['position'].isna()]
print(null_positions)

duplicate count: 0
season       0
round        0
driver_id    0
position     3
points       0
wins         0
dtype: int64
      season  round        driver_id  position  points  wins
2908    2015     19  kevin_magnussen       NaN     0.0     0
2956    2017     20           button       NaN     0.0     0
2957    2017     20            resta       NaN     0.0     0


DATA CLEANING: QUALIFYING

In [138]:
# Clean the driver qualifying data

qualifying_table = dataframes["qualifying"]
qualifying_table.head()

,race_id,driver_id,constructor_id,position,q1,q2,q3
0,1994_1,senna,williams,1,1:15.962,NaN,NaN
1,1994_1,michael_schumacher,benetton,2,1:16.290,NaN,NaN
2,1994_1,alesi,ferrari,3,1:17.385,NaN,NaN
3,1994_1,damon_hill,williams,4,1:17.554,NaN,NaN
4,1994_1,frentzen,sauber,5,1:17.806,NaN,NaN


In [139]:
# Split the race id into 2 (season, round)
qualifying_table[['season', 'round']] = qualifying_table['race_id'].str.split('_', expand=True)

# convert datatype of season and round to int
qualifying_table['season'] = qualifying_table['season'].astype(int)
qualifying_table['round'] = qualifying_table['round'].astype(int)

# filter the table so that only 2015-2025 seasons appear 
filtered_qualifying = qualifying_table[qualifying_table['season'] >= 2015]

# Data only until 5th round of 2025 season
filtered_qualifying

,race_id,driver_id,constructor_id,position,q1,q2,q3,season,round
1917,2015_1,hamilton,mercedes,1,1:28.586,1:26.894,1:26.327,2015,1
1918,2015_1,rosberg,mercedes,2,1:28.906,1:27.097,1:26.921,2015,1
1919,2015_1,massa,williams,3,1:29.246,1:27.895,1:27.718,2015,1
1920,2015_1,vettel,ferrari,4,1:29.307,1:27.742,1:27.757,2015,1
1921,2015_1,raikkonen,ferrari,5,1:29.754,1:27.807,1:27.790,2015,1
...,...,...,...,...,...,...,...,...,...
3012,2025_5,stroll,aston_martin,16,1:28.645,NaN,NaN,2025,5
3013,2025_5,doohan,alpine,17,1:28.739,NaN,NaN,2025,5
3014,2025_5,hulkenberg,sauber,18,1:28.782,NaN,NaN,2025,5
3015,2025_5,ocon,haas,19,1:29.092,NaN,NaN,2025,5


In [140]:
# Check datatypes of the columns to verify 
datatypes = filtered_qualifying.dtypes

# print datatypes of each column 
datatypes

race_id             str
driver_id           str
constructor_id      str
position          int64
q1                  str
q2                  str
q3                  str
season            int64
round             int64
dtype: object

In [141]:
# check for duplicates 
count_of_duplicates = filtered_qualifying.duplicated().sum()
print(f"duplicate count: {count_of_duplicates}")

# check for null values 
count_null_values = filtered_qualifying.isna().sum()
print(count_null_values)


duplicate count: 0
race_id             0
driver_id           0
constructor_id      0
position            0
q1                 15
q2                277
q3                562
season              0
round               0
dtype: int64


CLEANING THE DATA: RACES

In [142]:
# Clean the driver qualifying data

races_table = dataframes["races"]
races_table.head()

,race_id,season,round,race_name,date,time,circuit_id
0,1950_1,1950,1,British Grand Prix,1950-05-13,NaN,silverstone
1,1950_2,1950,2,Monaco Grand Prix,1950-05-21,NaN,monaco
2,1950_3,1950,3,Indianapolis 500,1950-05-30,NaN,indianapolis
3,1950_4,1950,4,Swiss Grand Prix,1950-06-04,NaN,bremgarten
4,1950_5,1950,5,Belgian Grand Prix,1950-06-18,NaN,spa


In [143]:
# filter the table so that only 2015-2025 races appear 
filtered_races = races_table[races_table['season'] >= 2015]
filtered_races

# make the date column into date time
filtered_races['date'] = pd.to_datetime(filtered_races['date'])

# remove the time column 
filtered_races = filtered_races.drop(columns=['time'])
filtered_races

,race_id,season,round,race_name,date,circuit_id
916,2015_1,2015,1,Australian Grand Prix,2015-03-15,albert_park
917,2015_2,2015,2,Malaysian Grand Prix,2015-03-29,sepang
918,2015_3,2015,3,Chinese Grand Prix,2015-04-12,shanghai
919,2015_4,2015,4,Bahrain Grand Prix,2015-04-19,bahrain
920,2015_5,2015,5,Spanish Grand Prix,2015-05-10,catalunya
...,...,...,...,...,...,...
1144,2025_20,2025,20,Mexico City Grand Prix,2025-10-26,rodriguez
1145,2025_21,2025,21,São Paulo Grand Prix,2025-11-09,interlagos
1146,2025_22,2025,22,Las Vegas Grand Prix,2025-11-23,vegas
1147,2025_23,2025,23,Qatar Grand Prix,2025-11-30,losail


In [144]:
# Check datatypes of the columns to verify 
datatypes = filtered_races.dtypes

# print datatypes of each column 
datatypes

race_id                  str
season                 int64
round                  int64
race_name                str
date          datetime64[us]
circuit_id               str
dtype: object

In [145]:
# check for duplicates 
count_of_duplicates = filtered_races.duplicated().sum()
print(f"duplicate count: {count_of_duplicates}")

# check for null values 
count_null_values = filtered_races.isna().sum()
print(count_null_values)


duplicate count: 0
race_id       0
season        0
round         0
race_name     0
date          0
circuit_id    0
dtype: int64


CLEANING THE DATA: RESULTS

In [146]:
# Clean the driver qualifying data

race_results_table = dataframes["results"]
race_results_table.head()

,race_id,driver_id,constructor_id,grid,position,position_order,points,laps,status
0,1950_1,farina,alfa,1,1,1,9.0,70,Finished
1,1950_1,fagioli,alfa,2,2,2,6.0,70,Finished
2,1950_1,reg_parnell,alfa,4,3,3,4.0,70,Finished
3,1950_1,cabantous,lago,6,4,4,3.0,68,+2 Laps
4,1950_1,rosier,lago,9,5,5,2.0,68,+2 Laps


In [147]:
# Split the race id into 2 (season, round)
race_results_table[['season', 'round']] = race_results_table['race_id'].str.split('_', expand=True)

# set the datatype to int for season and round
race_results_table['season'] = race_results_table['season'].astype(int)
race_results_table['round'] = race_results_table['round'].astype(int)
race_results_table['points'] = race_results_table['points'].astype(int)

race_results_table

,race_id,driver_id,constructor_id,grid,position,position_order,points,laps,status,season,round
0,1950_1,farina,alfa,1,1,1,9,70,Finished,1950,1
1,1950_1,fagioli,alfa,2,2,2,6,70,Finished,1950,1
2,1950_1,reg_parnell,alfa,4,3,3,4,70,Finished,1950,1
3,1950_1,cabantous,lago,6,4,4,3,68,+2 Laps,1950,1
4,1950_1,rosier,lago,9,5,5,2,68,+2 Laps,1950,1
...,...,...,...,...,...,...,...,...,...,...,...
7595,2025_5,stroll,aston_martin,16,16,16,0,49,Lapped,2025,5
7596,2025_5,doohan,alpine,17,17,17,0,49,Lapped,2025,5
7597,2025_5,bortoleto,sauber,20,18,18,0,49,Lapped,2025,5
7598,2025_5,tsunoda,red_bull,8,R,19,0,1,Retired,2025,5


In [148]:
filtered_race_results = race_results_table[race_results_table['season'] >= 2015]
filtered_race_results


,race_id,driver_id,constructor_id,grid,position,position_order,points,laps,status,season,round
6500,2015_1,hamilton,mercedes,1,1,1,25,58,Finished,2015,1
6501,2015_1,rosberg,mercedes,2,2,2,18,58,Finished,2015,1
6502,2015_1,vettel,ferrari,4,3,3,15,58,Finished,2015,1
6503,2015_1,massa,williams,3,4,4,12,58,Finished,2015,1
6504,2015_1,nasr,sauber,10,5,5,10,58,Finished,2015,1
...,...,...,...,...,...,...,...,...,...,...,...
7595,2025_5,stroll,aston_martin,16,16,16,0,49,Lapped,2025,5
7596,2025_5,doohan,alpine,17,17,17,0,49,Lapped,2025,5
7597,2025_5,bortoleto,sauber,20,18,18,0,49,Lapped,2025,5
7598,2025_5,tsunoda,red_bull,8,R,19,0,1,Retired,2025,5


In [149]:
# Check datatypes of the columns to verify 
datatypes = filtered_race_results.dtypes

# print datatypes of each column 
datatypes

race_id             str
driver_id           str
constructor_id      str
grid              int64
position            str
position_order    int64
points            int64
laps              int64
status              str
season            int64
round             int64
dtype: object

In [150]:
# check for duplicates 
count_of_duplicates = filtered_race_results.duplicated().sum()
print(f"duplicate count: {count_of_duplicates}")

# check for null values 
count_null_values = filtered_race_results.isna().sum()
print(count_null_values)

duplicate count: 0
race_id           0
driver_id         0
constructor_id    0
grid              0
position          0
position_order    0
points            0
laps              0
status            0
season            0
round             0
dtype: int64
